In [1]:
import sys
sys.path.append("../ingestion/python/src")
sys.path.append("../ingestion/python/LangGraph_Agent")

from utils import *
from langgraph.graph import StateGraph, START, END
from IPython.display import Image, display


from APIendpoint import PlacesAPI
from database import Database
import pandas as pd
import os
from dotenv import load_dotenv
load_dotenv(override=True)

db = Database()

c:\Users\rolan\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\langgraph\cache\base\__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [ ]:
# 1. Exécution des requêtes individuelles
bronze_company_null = db.execute("""
    SELECT COUNT(*) AS bronze_company_null
    FROM raw.job_offer
    WHERE company IS NULL OR company = '';
""")

resolved_by_llm = db.execute("""
    SELECT COUNT(*) AS resolved_by_llm
    FROM raw.job_offer b
    JOIN staging.enriched_offers s ON s.id_offer = b.id_job
    WHERE (b.company IS NULL OR b.company = '')
      AND s.raw_result->>'company_name' IS NOT NULL
      AND s.raw_result->>'company_name' NOT IN ('', 'null', 'Empresa confidencial');
""")

still_null_after_llm = db.execute("""
    SELECT COUNT(*) AS still_null_after_llm
    FROM raw.job_offer b
    JOIN staging.enriched_offers s ON s.id_offer = b.id_job
    WHERE (b.company IS NULL OR b.company = '')
      AND (s.raw_result->>'company_name' IS NULL
           OR s.raw_result->>'company_name' IN ('', 'null', 'Empresa confidencial'));
""")

# Ces 149 sont-elles concentrées sur une source (careerjet vs jsearch) ?
llm_resolved_details = db.execute("""
SELECT
    b.api_source,
    COUNT(*) AS n
FROM raw.job_offer b
JOIN staging.enriched_offers s ON s.id_offer = b.id_job
WHERE (b.company IS NULL OR b.company = '')
  AND (s.raw_result->>'company_name' IS NULL
       OR s.raw_result->>'company_name' IN ('', 'null', 'Empresa confidencial'))
GROUP BY b.api_source;
""")

# 2. Regroupement et affichage de tous les résultats
r = {
    "bronze_company_null": bronze_company_null,
    "resolved_by_llm": resolved_by_llm,
    "still_null_after_llm": still_null_after_llm,
    "llm_resolved_details": llm_resolved_details
}

print(r)


In [3]:
q = """SELECT
    b.id_job,
    LENGTH(b.offer_description) AS description_length,
    b.collected_at
FROM raw.job_offer b
JOIN staging.enriched_offers s ON s.id_offer = b.id_job
WHERE b.api_source = 'careerjet'
  AND (b.company IS NULL OR b.company = '')
  AND (s.raw_result->>'company_name' IS NULL
       OR s.raw_result->>'company_name' IN ('', 'null', 'Empresa confidencial'))
ORDER BY b.collected_at
LIMIT 20;"""

r = db.execute(q)
r

[{'id_job': 'cj_c54e2eeee7d8ae0c7ae109464f711008',
  'description_length': 263,
  'collected_at': datetime.datetime(2026, 6, 9, 22, 45, 48, 488360, tzinfo=datetime.timezone.utc)},
 {'id_job': 'cj_59cf47c22d3120c4c544a4cabfc915ca',
  'description_length': 269,
  'collected_at': datetime.datetime(2026, 6, 9, 22, 45, 48, 488475, tzinfo=datetime.timezone.utc)},
 {'id_job': 'cj_fef20ce3a0164e2b99fa0c3c3fdd577a',
  'description_length': 262,
  'collected_at': datetime.datetime(2026, 6, 9, 22, 45, 48, 488516, tzinfo=datetime.timezone.utc)},
 {'id_job': 'cj_19971103e452c44dd99b69afe7026187',
  'description_length': 306,
  'collected_at': datetime.datetime(2026, 6, 9, 22, 45, 48, 488608, tzinfo=datetime.timezone.utc)},
 {'id_job': 'cj_fdd47bba11be1f542a3df663db7324ee',
  'description_length': 260,
  'collected_at': datetime.datetime(2026, 6, 9, 22, 45, 48, 488628, tzinfo=datetime.timezone.utc)},
 {'id_job': 'cj_48afdfe1187458d5f203eda5f2c2609f',
  'description_length': 272,
  'collected_at': da